In [ ]:
import sys
import numpy as np
import scipy.ndimage as ndi

In [ ]:
sys.path.append("..")
from wavelet_transform.utils.save_plots import save_figure

from wavelet_transform.postprocess import plotting as plot
import wavelet_transform.utils.kmeans_results as results
from wavelet_transform.utils.extensions import trim_data, psd, to_dB
import wavelet_transform.utils.statistical_features as sf

In [ ]:
savefigs_flag = True
Font_Size = 24
Tick_Size = 24
Fig_Height = 400
Fig_Width = 800
fft_lim = 10

Bubble dynamics results

In [ ]:
# # Get bubble dynamics results
# frequency = 2e6


# exp_ids = [f"{R_init}_{frequency}_{pac}" for pac, R_init in [ (1e6, 15e-6),(1e6, 1e-6)]]

# # R_init =1e-6
# # pac = 1e6
# # exp_ids = [f"{R_init}_{frequency}_{pac}"]
# # print(exp_ids)
# exp_results = [results.get_data(result_id) for result_id in exp_ids]

# exp_labels = ["Stable", "Transient"]  # Legend for final plot

# label_maps = [
#     {0: "measurement noise", 1: "fundamental"},
#     {0:  "transient", 1:"fundamental and harmonics"}
# ]

# # label_maps = [
# #     {0: "broadband", 1: "measurement noise", 2: "fundamental"},
# #     {0: "broadband", 1: "fundamental", 2: "measurement noise", 3:"??"}
# # ]



Results for other datasets

In [ ]:
# # Get results from experiment number
# exp_number = 9  # 9, 15, 55
# result_id = str(exp_number)
# frequency = 2e6

# exp_ids = [9, 15, 55]
# exp_results = [results.get_data(result_id) for result_id in exp_ids]
# label_maps = [
#     {0: "broadband", 1: "measurement", 2:"fundamental"},
#     {0: "broadband", 1: "harmonic", 2:"measurement", 3: "fundamental", 4:"transient"},
#     {0: "measurement", 1: "broadband", 2: "fundamental"}
# ]

In [ ]:
# Get PCD results
voltage = 280
pcd_number = 1
no_clusters = 3, 6
result_id = f"{voltage}_{pcd_number}_{no_clusters[0]}_{no_clusters[1]}"
frequency = 1.1e6
period = 1/frequency
exp_results = results.get_data(result_id)

# label_map = {1:"noise", 0:"harm/bb", 2:"fund", 3:"bb"}
# label_map = {1:"noise", 0:"harm/bb", 2:"bb", 3:"fund.", 4:"sub/bb"}


In [ ]:
# Keep track of all results for combined plot in next cell

required_info = ["data", "tr_obj", "centers", "labels", "sampling rate", "freq_threshold",
                 "centers_sorted", "labels_sorted"]
# Plot separate clusterig results
# unpack results
data_train, tr_obj, centers, labels, sampling_rate, freq_threshold, centers_sorted, labels_sorted= (
    tuple(exp_results[i] for i in required_info)
)

if no_clusters==2:
    if centers['average energy'][0] > centers['average energy'][1]:
        label_map = {0: "periodic", 1:"transient"}
    else: 
        label_map = {1: "periodic", 0:"transient"}
else:
    label_map = dict((j, f"Cluster{j}") for j in range(len(centers)))

classifying_metrics = data_train.columns.to_list()
print(label_map)
fig = plot.plot_clustering(
    data_train, centers_sorted, labels_sorted, 
    metrics_to_plot=classifying_metrics,
    title=None, 
    label_map=label_map
)

if savefigs_flag == True:
    save_figure(fig, f"clustering_{result_id}")

In [ ]:
# Plot centroid metrics
fig = plot.plot_centers(
    classifying_metrics, 
    centers_sorted,
    label_map=dict((i, str(i)) for i in range(len(centers_sorted))),
    xlabel_user="Cluster"
)
if savefigs_flag == True:
    save_figure(fig, f"sig_components_metrics_{result_id}")

- change legend to stable and transient
- add error bars for multiple repetitions of the same experiment

In [ ]:
# Set clusters to None to plot all of them, otherwise, plot a list of int
fig = plot.plot_fft_by_group(
    tr_obj, 
    labels_sorted, 
    sampling_rate=sampling_rate,
    clusters=None, 
    xlims=[0,fft_lim],
    ylims=[-100,50],
    fundamental_freq=frequency,
    freq_threshold=freq_threshold, 
    label_map=label_map,
    fig_height=Fig_Height,
    # fig_width=Fig_Width,
    title=""
)
if savefigs_flag == True:
    save_figure(fig, f"fft_rec_centers_{result_id}")

# to plot the time axis normalised by the period, uncomment the following block
# t_plot_dic = {"t/T [DL]":np.linspace(0,(len(tr_obj.data)/sampling_rate)/period,len(tr_obj.data) )}
# fig = plot.plot_reconstructions_by_group(
#    tr_obj, labels, t_plot=t_plot_dic,
#     freq_threshold=freq_threshold, label_map=label_map
# )

fig = plot.plot_reconstructions_by_group(
    tr_obj, 
    labels_sorted, 
    t_plot=sampling_rate,
    freq_threshold=freq_threshold, 
    label_map=label_map,
    fig_height=Fig_Height,
    # fig_width=Fig_Width
)
if savefigs_flag == True:
    save_figure(fig, f"sig_rec_centers_{result_id}")



In [ ]:
import wavelet_transform.wpt.inverse_transform as inv
from wavelet_transform.utils.preprocessing import preprocess
data_raw_preprocessed = preprocess(exp_results['raw_data'], window=False)

def get_reconstructions_by_group(tr_obj, labels, freq_threshold):
    signal_length = len(tr_obj.data)
    n_clusters = len(set(labels))
    # List reconstructions by group
    reconstructions = [np.zeros(signal_length) for _ in range(n_clusters)]

    # Sum all reconstructions belonging to the same group
    for node, assignment in zip(
        tr_obj.get_leaf_nodes(freq_threshold=freq_threshold), labels
    ):  # natural order
        rec_signal = inv.inv_dwpt(tr_obj, level=node.level, nodes=node)
        i = assignment  # shorten name for readability
        reconstructions[i] = reconstructions[i] + rec_signal

    return reconstructions

In [ ]:
reconstructions = get_reconstructions_by_group(tr_obj, labels_sorted, freq_threshold=freq_threshold)

In [ ]:
trunc_period = (30*period, 40*period)
data_trunc, t_data_trunc = [], []

data_trunc_tmp, t_data_trunc_tmp = trim_data(data_raw_preprocessed, 
                                             np.linspace(0,len(data_raw_preprocessed)/sampling_rate,len(data_raw_preprocessed)),
                                             trunc_period, return_time=True)
data_trunc.append(data_trunc_tmp)
t_data_trunc.append(t_data_trunc_tmp)

for i in range(len(reconstructions)):
    data_trunc_tmp, t_data_trunc_tmp = trim_data(reconstructions[i], 
                                     np.linspace(0,len(reconstructions[i])/sampling_rate,len(reconstructions[i])),
                                     trunc_period, return_time=True)
    data_trunc.append(data_trunc_tmp)
    t_data_trunc.append(t_data_trunc_tmp)

legend = ['Original signal'] + [label_map[i] for i in range(len(centers))] 

import plotly.graph_objects as go
# from wavelet_transform.wpt.tree_graph import TreeGraph

from itertools import cycle, islice
lines = ['solid','dot', 'dashdot', 'solid','dot', 'dash',  'longdash','longdashdot']
# colours = TreeGraph.colour_dict  # Get colours for each cluster from tree graph
colours = ['blue','orange','brown','green','purple','cyan','red']
linestyles = cycle(lines)
colournames = cycle(colours)

fsize = 20  # font size
tick_size = 16 # tick size
cl_label_fsize = 14 # clusters labels font size
fig_ae = go.Figure()

for i in range(len(data_trunc)):

    # to plot the time axis normalised by the period, uncomment the following block
    ## fig_ae.add_trace( 
    ##     go.Scatter(x=t_data_trunc[i]/period, y=data_trunc[i], 
    ##                line=dict(dash=next(linestyles),color=next(colournames)), name=legend[i])
    ## )

    fig_ae.add_trace( 
        go.Scatter(x=t_data_trunc[i]*1e6, y=data_trunc[i], 
                   line=dict(dash=next(linestyles),color=next(colournames)), name=legend[i])
    )
fig_ae.update_layout(
    height=400,
    width=1000,
    font_size = fsize,
    xaxis=dict(title="t [&mu;s]",tickfont = dict(size=tick_size)),
    yaxis=dict(title="Amplitude",tickfont = dict(size=tick_size)),
    plot_bgcolor='white',
    legend=dict(
        x=1.01,
        y=1.,
        traceorder='normal',
        font=dict(size=tick_size),
    ),
    legend_title=None
)
fig_ae.update_xaxes(
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey',
)
fig_ae.update_yaxes(
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey',
)
fig_ae.show()

if savefigs_flag == True:
    save_figure(fig_ae, f"sig_rec_centers_truncated_{result_id}")
    

In [ ]:
label_map

In [ ]:
tr_obj.calc_frequencies(sampling_rate)
vmin = -2
vmax = 2
fig = plot.heisenberg(
    tr_obj, 1/sampling_rate*len(tr_obj.data), max_freq=.1, padding="repeat",
    vmin=vmin, vmax=vmax, color_map='viridis'
)
if savefigs_flag is True:
    save_figure(fig, f"heisenberg_{result_id}")

In [ ]:
# from wavelet_transform.utils.preprocessing import preprocess
# preprocess(exp_results['raw_data'], window=False)

In [ ]:
colours

In [ ]:
from wavelet_transform.utils.statistical_features import kurtosis

reconstructions = plot.get_reconstructions_by_group(tr_obj, labels_sorted, freq_threshold)
for i, rec in enumerate(reconstructions):
    print(f"kurtosis of reconstructed cluster {i}: {kurtosis(rec)}")

In [ ]:
from scipy import signal
import matplotlib.pyplot as plt

def stft_plt(data,frequency, period, sampling_rate):
    fig, ax = plt.subplots()
    fig.set_size_inches(18, 9)
    f_stft, t_stft, mag_stft = signal.stft(data, sampling_rate, nperseg=2**11, detrend=False, scaling='spectrum')
    stft_plt = ax.pcolormesh(t_stft/period, f_stft/frequency, 20*np.log10(np.abs(mag_stft)), vmin=-100, vmax=1, shading='gouraud')
    ax.set_ylabel("$f / f_0$ [DL]", fontsize=fsize)
    ax.set_xlabel("t/T [DL]", fontsize=fsize)  # noqa
    ax.set_title('STFT - spectrum', fontsize=fsize)
    ax.set_ylim([0,fft_lim]) 
    plt.xticks(fontsize=tick_size)
    plt.yticks(np.arange(0,11),np.arange(0,11),fontsize=tick_size)
    cbar_label = ("Magnitude [dB]")
    fig.tight_layout()
    cbar = fig.colorbar(stft_plt, pad=0.01)
    cbar.ax.tick_params(labelsize=tick_size)
    cbar.set_label(label=cbar_label, size=fsize)
    plt.show()

In [ ]:
for i in range(len(centers)):
    stft_plt(reconstructions[i],frequency, period, sampling_rate)

In [ ]:
def get_recs_from_cluster_and_threshold(
        centers_sorted, labels_sorted, freq_threshold, threshold, tr_obj, inv_keep_ratio
):
    plot_centres = np.arange(len(centers_sorted)) 
    plot_centres = plot_centres[centers_sorted['kurtosis'] > threshold]
    reconstructions = [np.zeros(len(tr_obj.data)) for _ in range(2)]

    for (node, assignment) in zip(
        tr_obj.get_leaf_nodes(freq_threshold=freq_threshold), labels_sorted
    ):
        if int(assignment) in plot_centres:
            # List reconstructions
            rec_signal = inv.inv_dwpt(tr_obj, node.level, nodes=node)
            j = 0 if sf.kurtosis(trim(rec_signal, inv_keep_ratio)) > threshold else 1
            reconstructions[j] = reconstructions[j] + rec_signal

    return reconstructions

def trim(data, ratio):
    lim = int((1 - ratio) / 2 * len(data))
    if lim == 0:
        return data
    return data[lim:-lim]

In [ ]:
kurt_threshold = 3
reconstructions = get_recs_from_cluster_and_threshold(
    centers_sorted, labels_sorted, freq_threshold, kurt_threshold, tr_obj, inv_keep_ratio=0.9
)

psd_rec_higher, freq_higher = psd(reconstructions[0], sampling_rate, frequency)
psd_rec_lower, freq_lower = psd(reconstructions[1], sampling_rate, frequency)

psd_curve = psd_curve = ndi.gaussian_filter1d(psd_rec_higher, 3)

rms = np.sqrt(np.mean(psd_curve**2))
half_peak = max(psd_curve) / 2
e_peak = max(psd_curve) / np.e**2

broadband_lines = {
    "rms": rms, 
    "FWHM": half_peak, 
    "1/e<sup>2</sup> peak": e_peak
}

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=freq_higher,
        y=to_dB(psd_rec_higher),
        mode="lines",
        name="higher than threshold",
        line=dict(dash="longdash")
    )
)
# fig.add_trace(
#     go.Scatter(
#         x=freq_lower,
#         y=to_dB(psd_rec_lower),
#         mode="lines",
#         name="lower than threshold",
#     )
# )
fig.add_trace(
    go.Scatter(
        x=freq_higher,
        y=to_dB(psd_curve),
        name="envelope"
    )
)

for line_name, line in broadband_lines.items():
    db_line = 10 * np.log10(line)
    fig.add_trace(
        go.Scatter(
            x=freq_higher,
            y=db_line*np.ones(len(freq_higher)),
            line=dict(dash="dash"),
            name=line_name
        )
    )
fig.update_layout(
    xaxis=dict(title="f / f<sub>0</sub> [DL]", range=[0, 10]),
    yaxis=dict(title="Magnitude [dB]")
)
fig.update_layout(
    height=400,
    width=800,
    plot_bgcolor='white',
    legend=dict(
        x=1.01,
        y=1.,
        traceorder='normal',
        font=dict(size=tick_size),
    ),
    legend_title=None
)
fig.update_xaxes(
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey',
)
fig.update_yaxes(
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey',
)
fig.show()

if savefigs_flag is True:
    save_figure(fig, f"thresholding_{result_id}")

## High kurtosis plot 280mvpp, 80mvpp combined

Combined time series reconstructions from high kurtosis nodes in high kurtosis clusters

In [ ]:
result_id2 = results.get_data("280_1_3_6")
result_id1 = results.get_data("80_1_3_6")

In [ ]:
threshold = 3  # kurtosis threshold
inv_keep_ratio = 0.9

reconstructions1 = get_recs_from_cluster_and_threshold(
    result_id1["centers_sorted"],
    result_id1["labels_sorted"],
    result_id1["freq_threshold"],
    threshold,
    result_id1["tr_obj"],
    inv_keep_ratio
)

reconstructions2 = get_recs_from_cluster_and_threshold(
    result_id2["centers_sorted"],
    result_id2["labels_sorted"],
    result_id2["freq_threshold"],
    threshold,
    result_id2["tr_obj"],
    inv_keep_ratio
)

data_len = len(tr_obj.data)
t_plot = np.linspace(0, 1/sampling_rate*data_len, data_len) * 1e6
# Add number to plot_centres to plot reconstructions of all nodes belonging to the cluster
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=reconstructions2[0],
        mode="lines",
        name="D2",
    )
)
fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=reconstructions1[0],
        mode="lines",
        name="D1",
    )
)
fig.update_layout(
    xaxis_title="time [s]",
    yaxis_title="P<sub>rad</sub>/P<sub>0</sub> [DL]"
)

fig.update_layout(
    height=400,
    width=800,
    font_size = fsize,
    xaxis=dict(title="t [&mu;s]",tickfont = dict(size=tick_size)),
    yaxis=dict(title="Amplitude",tickfont = dict(size=tick_size)),
    plot_bgcolor='white',
    legend=dict(
        x=1.01,
        y=1.,
        traceorder='normal',
        font=dict(size=tick_size),
    ),
    legend_title=None
)
fig.update_xaxes(
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey',
)
fig.update_yaxes(
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey',
)
fig.show()

if savefigs_flag is True:
    save_figure(fig, "Reconstructed_thresholded_clusters_80_and_280_3_6")


In [ ]:
d1_rec = reconstructions1[0]
d2_rec = reconstructions2[0]
d1_std = np.std(result_id1["data_trim"])
d2_std = np.std(result_id2["data_trim"])

for n, (i, j) in enumerate(zip([d1_rec, d2_rec], [d1_std, d2_std])):
    n += 1
    print(f"energy d{n}: {sf.energy(i) * j**2}")    
    print(f"kurtosis d{n}: {sf.kurtosis(i)}")
    print(f"skewness d{n}: {sf.skewness(i)}")
    print(f"spectral entropy d{n}: {sf.spectral_entropy(i)}")